### Conhecendo o Schema e primeiros registos

In [0]:
df = spark.read.parquet('/Volumes/taxi_nyc/raw/landing/yellow/yellow_tripdata_2024-01.parquet') 
df.display()

## Displays the show([n], truncate=False) : Displays the first n rows in a tabular format (default is 20). Use truncate=False to show full column values.
# On databricks, display() has a better table view



In [0]:
## Entendendo a estrutura dos dados
df.printSchema()

In [0]:
## Print de todas as colunas e seus tipos
df.dtypes


### Estatisticas do dado

In [0]:
## Count(): Triggers a full scan of the DataFrame to return the total number of rows. 
# It is an action and can be expensive on large data.

print("Quantidade de linhas:  ---> ", df.count())
print("Quantidade de colunas: ---> ", len(df.columns))

### Estatisticas descritivas

In [0]:
df.describe().show(truncate=False) # Resumo estatístico das colunas numéricas (count, mean, stddev, min, max)
# df.summary().show() --> versão mais completa      (inclui % e mais métricas)

##Valores de amount negativos

In [0]:
df.select('*').where(df['total_amount'] < 0).groupBy('payment_type').count().show()
df.select('*').where(df['total_amount'] < 0).show(10)
df.select('*').where((df['total_amount'] < 0) & (df['fare_amount'] > 0)).show()


### Checagem de nulos:

In [0]:
from pyspark.sql.functions import col, count, when, round, lit, sum, avg, max, min

# Contagem de nulos por coluna
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

# Com percentual
total = df.count()
df.select([
    round((count(when(col(c).isNull(), c)) / total) * 100, 2).alias(c)
    for c in df.columns
]).show()

In [0]:
# Analise de linhas onde QUALQUER coluna é nula
from pyspark.sql.functions import col
from functools import reduce

condicao = reduce(
    lambda a, b: a | b,
    [col(c).isNull() for c in df.columns]
)

df.filter(condicao).show(10)

In [0]:
from pyspark.sql.functions import col, count, when, round as _round
from functools import reduce

# Deixe as colunas de agrupamento fora da condição,
# senão um nulo nelas mesmas mistura a análise
colunas_teste = [c for c in df.columns if c not in ('payment_type', 'VendorID')]

tem_nulo = reduce(lambda a, b: a | b, [col(c).isNull() for c in colunas_teste])

(
    df.groupBy('payment_type', 'VendorID')
      .agg(
          count('*').alias('total'),
          count(when(tem_nulo, 1)).alias('com_nulo')
      )
      .withColumn('pct_nulo', _round(col('com_nulo') / col('total') * 100, 2))
      .orderBy(col('com_nulo').desc())
      .show(50, truncate=False)
)

### Checagem de duplicatas

In [0]:
# Linhas totalmente duplicadas
total = df.count()
distintos = df.distinct().count()
duplicadas = total - distintos

print(f'Total de linhas: {total}')
print(f'Total de linhas distintas: {distintos}')
print(f'Total de linhas duplicadas: {duplicadas}')

### Distribuição de valores categóricos

In [0]:
from pyspark.sql.functions import countDistinct

df.select([
    countDistinct(c).alias(c)
    for c in df.columns
]).show()


### Valores por coluna específica (É muito caro de ser feito)



In [0]:

""""
df.groupBy("VendorID").count().show()
df.groupBy("passenger_count").count().show()
df.groupBy("store_and_fwd_flag").count().show()
df.groupBy("RatecodeID").count().show()
df.groupBy("payment_type").count().show()
df.groupBy("Airport_fee").count().show()
"""


### Valores min e max

In [0]:

# Checando o range de datas
from pyspark.sql.functions import min, max

df.select(
    min("tpep_pickup_datetime").alias("min_date_pickup"),
    max("tpep_pickup_datetime").alias("max_date_pickup"),
    min("tpep_dropoff_datetime").alias("min_date_dropoff"),
    max("tpep_dropoff_datetime").alias("max_date_dropoff")
).show()

### Checagem espaços em branco e string vazias

In [0]:
"""
from pyspark.sql.functions import trim, length

# Diferença entre nulo, vazio ("") e só espaços ("   ")
df.select([
    count(when(
        col(c).isNull() | (trim(col(c)) == " ") | (length(trim(col(c))) == 0), c
    )).alias(c)
    for c in df.columns
    if df.schema[c].dataType.simpleString() == "string"
]).show()
"""

### Partições

In [0]:
"""

# Quantas partições o DataFrame tem
print("Número de partições:", df.rdd.getNumPartitions())

"""